# Base Model - Time Series Regression

## 準備

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# %pip install -q catboost

In [ ]:
# ====================================================
# Library
# ====================================================
import os
import gc
import warnings

warnings.filterwarnings("ignore")
import random
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import lightgbm as lgb
import xgboost as xgb
from catboost import Pool, CatBoostRegressor

In [ ]:
# ====================================================
# Configurations
# ====================================================
# 実行環境に合わせて変更してください。
# 例: MAIN_PATH = Path("/content/drive/MyDrive/kaggle/bike-sharing-demand")
MAIN_PATH = Path.cwd()

class CFG:
    VER = 1
    AUTHOR = "Jun-Morita"
    COMPETITION = "bike-sharing-demand"
    MAIN_PATH = MAIN_PATH
    DATA_PATH = MAIN_PATH / "data"
    OOF_DATA_PATH = MAIN_PATH / "oof"
    MODEL_DATA_PATH = MAIN_PATH / "models"
    METHOD_LIST = ["lightgbm", "xgboost", "catboost", "linear"]
    METHOD_WEIGHT_DICT = {"lightgbm": 0.3, "xgboost": 0.3, "catboost": 0.3, "linear": 0.1}
    #USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SEEDS = 1  # 初回実行を軽くする。精度重視なら 5 などに増やす。
    N_SPLIT = 3
    VALID_MONTHS = 3
    datetime_col = "datetime"
    raw_target_col = "count"
    target_col = "count_log1p"
    metric = "rmsle"
    metric_maximize_flag = False
    BASE_DATETIME = None

    num_boost_round = 1000
    early_stopping_round = 50
    verbose = 100

    lgb_params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 12,
        "seed": SEED,
        "verbosity": -1,
    }

    xgb_params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "learning_rate": 0.05,
        "max_depth": 4,
        "seed": SEED,
    }

    cat_params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "learning_rate": 0.05,
        "iterations": num_boost_round,
        "depth": 4,
        "random_seed": SEED,
    }

In [ ]:
# 実行前チェック
for path in [CFG.DATA_PATH, CFG.OOF_DATA_PATH, CFG.MODEL_DATA_PATH]:
    path.mkdir(parents=True, exist_ok=True)

for file_name in ["train.csv", "test.csv", "sampleSubmission.csv"]:
    if not (CFG.DATA_PATH / file_name).exists():
        raise FileNotFoundError(
            f"{CFG.DATA_PATH / file_name} が見つかりません。"
            " MAIN_PATH をデータ配置先に合わせて変更してください。"
        )

In [ ]:
# ====================================================
# Seed everything
# ====================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG.SEED)

In [ ]:
# ====================================================
# Timer
# ====================================================
from time import time


class Timer:
    def __init__(self, logger=None, format_str="{:.3f}[s]", prefix=None, suffix=None, sep=" "):
        if prefix:
            format_str = str(prefix) + sep + format_str
        if suffix:
            format_str = format_str + sep + str(suffix)
        self.format_str = format_str
        self.logger = logger
        self.start = None
        self.end = None

    @property
    def duration(self):
        if self.end is None:
            return 0
        return self.end - self.start

    def __enter__(self):
        self.start = time()

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time()
        out_str = self.format_str.format(self.duration)
        if self.logger:
            self.logger.info(out_str)
        else:
            print(out_str)

In [ ]:
def to_original_scale(y_log):
    return np.maximum(np.expm1(np.asarray(y_log, dtype=float)), 0)


def calc_score(y_true, y_pred):
    y_true_count = to_original_scale(y_true)
    y_pred_count = to_original_scale(y_pred)
    return np.sqrt(mean_squared_log_error(y_true_count, y_pred_count))


def calculate_regression_metrics(y_true, y_pred):
    y_true_count = to_original_scale(y_true)
    y_pred_count = to_original_scale(y_pred)
    return {
        CFG.metric: calc_score(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true_count, y_pred_count)),
        "mae": mean_absolute_error(y_true_count, y_pred_count),
    }

## Read Data

In [ ]:
# ====================================================
# Read Data
# ====================================================
# 基本的なデータ
train_df = pd.read_csv(CFG.DATA_PATH / "train.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")
submission_df = pd.read_csv(CFG.DATA_PATH / "sampleSubmission.csv")

for df in [train_df, test_df, submission_df]:
    df[CFG.datetime_col] = pd.to_datetime(df[CFG.datetime_col])

train_df = train_df.sort_values(CFG.datetime_col).reset_index(drop=True)
test_df = test_df.sort_values(CFG.datetime_col).reset_index(drop=True)
submission_df = submission_df.sort_values(CFG.datetime_col).reset_index(drop=True)

train_df[CFG.target_col] = np.log1p(train_df[CFG.raw_target_col])
CFG.BASE_DATETIME = min(
    train_df[CFG.datetime_col].min(),
    test_df[CFG.datetime_col].min(),
)

print(f"train: {train_df.shape}")
print(f"test: {test_df.shape}")
print(f"train period: {train_df[CFG.datetime_col].min()} -> {train_df[CFG.datetime_col].max()}")
print(f"test period: {test_df[CFG.datetime_col].min()} -> {test_df[CFG.datetime_col].max()}")

## EDA

In [ ]:
display(train_df)
display(test_df)

In [ ]:
display(train_df.describe())
display(test_df.describe())

In [ ]:
display(train_df.isna().sum())
display(test_df.isna().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


daily_count_df = (
    train_df.set_index(CFG.datetime_col)[CFG.raw_target_col]
    .resample("D")
    .sum()
    .rename("count")
    .reset_index()
)
hourly_count_df = (
    train_df.assign(hour=train_df[CFG.datetime_col].dt.hour)
    .groupby("hour")[CFG.raw_target_col]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(daily_count_df[CFG.datetime_col], daily_count_df["count"])
axes[0].set_title("Daily Rental Count")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Count")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.lineplot(data=hourly_count_df, x="hour", y=CFG.raw_target_col, marker="o", ax=axes[1])
axes[1].set_title("Mean Rental Count by Hour")
axes[1].grid(True, linestyle="--", alpha=0.5)
fig.tight_layout()
plt.show()

## Preprocessing data

`casual` と `registered` は `count` の内訳で、test データには存在しません。
目的変数リークを避けるため、特徴量には使用しません。

In [ ]:
def create_weather_features(input_df):
    df = input_df.copy()
    return df[
        [
            "season",
            "holiday",
            "workingday",
            "weather",
            "temp",
            "atemp",
            "humidity",
            "windspeed",
        ]
    ]

In [ ]:
def create_calendar_features(input_df):
    df = input_df.copy()
    dt = pd.to_datetime(df[CFG.datetime_col])
    return pd.DataFrame(
        {
            "year": dt.dt.year,
            "month": dt.dt.month,
            "day": dt.dt.day,
            "dayofweek": dt.dt.dayofweek,
            "hour": dt.dt.hour,
        },
        index=df.index,
    )

In [ ]:
def create_cyclical_features(input_df):
    df = input_df.copy()
    dt = pd.to_datetime(df[CFG.datetime_col])
    return pd.DataFrame(
        {
            "hour_sin": np.sin(2 * np.pi * dt.dt.hour / 24),
            "hour_cos": np.cos(2 * np.pi * dt.dt.hour / 24),
            "dayofweek_sin": np.sin(2 * np.pi * dt.dt.dayofweek / 7),
            "dayofweek_cos": np.cos(2 * np.pi * dt.dt.dayofweek / 7),
            "month_sin": np.sin(2 * np.pi * (dt.dt.month - 1) / 12),
            "month_cos": np.cos(2 * np.pi * (dt.dt.month - 1) / 12),
        },
        index=df.index,
    )

In [ ]:
def create_trend_feature(input_df):
    df = input_df.copy()
    dt = pd.to_datetime(df[CFG.datetime_col])
    elapsed_hours = (dt - CFG.BASE_DATETIME).dt.total_seconds() / 3600
    return pd.DataFrame({"elapsed_hours": elapsed_hours}, index=df.index)

In [ ]:
from typing import List


def build_feature(input_df: pd.DataFrame, feature_functions: List) -> pd.DataFrame:
    # 出力するデータフレームを空で用意して
    out_df = pd.DataFrame()

    print("start build features...")

    # 各特徴生成関数ごとで
    for func in feature_functions:
        with Timer(prefix=f"create {func.__name__}"):
            # 特徴量を作成し
            _df = func(input_df)

        # 横方向 (axis=1) にがっちゃんこ (concat) する
        out_df = pd.concat([out_df, _df], axis=1)

    return out_df

In [ ]:
feature_functions = [
    create_weather_features,
    create_calendar_features,
    create_cyclical_features,
    create_trend_feature,
]

In [ ]:
feat_train_df = build_feature(input_df=train_df, feature_functions=feature_functions)
feat_test_df = build_feature(input_df=test_df, feature_functions=feature_functions)

train_full_df = pd.concat(
    [
        train_df[[CFG.datetime_col, CFG.raw_target_col, CFG.target_col]],
        feat_train_df,
    ],
    axis=1,
)

features = feat_train_df.columns.tolist()

categorical_features = [
    "season",
    "holiday",
    "workingday",
    "weather",
    "year",
    "month",
    "dayofweek",
    "hour",
]
linear_categorical_features = categorical_features
for col in categorical_features:
    full_cat = pd.concat([feat_train_df[col], feat_test_df[col]]).astype("category")
    train_full_df[col] = pd.Categorical(
        train_full_df[col],
        categories=full_cat.cat.categories,
    )
    feat_train_df[col] = pd.Categorical(
        feat_train_df[col],
        categories=full_cat.cat.categories,
    )
    feat_test_df[col] = pd.Categorical(
        feat_test_df[col],
        categories=full_cat.cat.categories,
    )

display(features)
display(train_full_df)
display(feat_test_df)

## Time Series Validation

時系列データでは未来の情報を使って過去を検証しないよう、ランダム KFold は使いません。
直近の連続した月を validation とし、それより前のデータだけで学習する expanding-window CV を
使用します。

Kaggle の test データには天候情報が含まれているため、この baseline では予測時点で既知の
カレンダー・天候特徴量だけを使用します。実運用では天候予報の入手可否も確認してください。

In [ ]:
def make_time_series_splits(input_df):
    month_series = input_df[CFG.datetime_col].dt.to_period("M")
    unique_months = np.array(sorted(month_series.unique()))
    n_validation_months = CFG.N_SPLIT * CFG.VALID_MONTHS
    if n_validation_months >= len(unique_months):
        raise ValueError("学習期間より validation 期間が長すぎます。")

    validation_months = unique_months[-n_validation_months:]
    splits = []
    for fold in range(CFG.N_SPLIT):
        start = fold * CFG.VALID_MONTHS
        end = start + CFG.VALID_MONTHS
        fold_validation_months = validation_months[start:end]
        train_index = np.flatnonzero(month_series < fold_validation_months[0])
        valid_index = np.flatnonzero(month_series.isin(fold_validation_months))
        splits.append((train_index, valid_index))
    return splits


def describe_time_series_splits(input_df, splits):
    rows = []
    for fold, (train_index, valid_index) in enumerate(splits, start=1):
        rows.append(
            {
                "fold": fold,
                "train_rows": len(train_index),
                "valid_rows": len(valid_index),
                "train_start": input_df.iloc[train_index][CFG.datetime_col].min(),
                "train_end": input_df.iloc[train_index][CFG.datetime_col].max(),
                "valid_start": input_df.iloc[valid_index][CFG.datetime_col].min(),
                "valid_end": input_df.iloc[valid_index][CFG.datetime_col].max(),
            }
        )
    return pd.DataFrame(rows)


time_series_splits = make_time_series_splits(train_full_df)
split_summary_df = describe_time_series_splits(train_full_df, time_series_splits)
display(split_summary_df)

In [ ]:
def plot_time_series_splits(split_summary_df):
    fig, ax = plt.subplots(figsize=(12, 4))
    for _, row in split_summary_df.iterrows():
        fold = int(row["fold"])
        ax.plot(
            [row["train_start"], row["train_end"]],
            [fold, fold],
            linewidth=8,
            label="train" if fold == 1 else None,
        )
        ax.plot(
            [row["valid_start"], row["valid_end"]],
            [fold, fold],
            linewidth=8,
            label="validation" if fold == 1 else None,
        )
    ax.set_yticks(split_summary_df["fold"])
    ax.set_ylabel("Fold")
    ax.set_title("Expanding-Window Time Series CV")
    ax.grid(True, axis="x", linestyle="--", alpha=0.5)
    ax.legend()
    fig.tight_layout()
    plt.show()


plot_time_series_splits(split_summary_df)

### Lag feature を追加するときの注意

目的変数から lag や rolling 統計量を作る場合は、必ず `shift()` を先に適用してください。
この competition の test 期間で lag 特徴量を使うには、予測値を順次履歴へ追加する
recursive forecast などの設計が必要です。そのため、以下は学習データでの例示に留め、
baseline の特徴量には追加しません。

In [ ]:
def create_lag_features_for_training_only(input_df):
    df = input_df.sort_values(CFG.datetime_col).copy()
    shifted_count = df[CFG.raw_target_col].shift(1)
    df["count_lag_1"] = df[CFG.raw_target_col].shift(1)
    df["count_lag_24"] = df[CFG.raw_target_col].shift(24)
    df["count_rolling_mean_24"] = shifted_count.rolling(24).mean()
    df["count_rolling_mean_168"] = shifted_count.rolling(24 * 7).mean()
    return df[
        [
            "count_lag_1",
            "count_lag_24",
            "count_rolling_mean_24",
            "count_rolling_mean_168",
        ]
    ]


lag_feature_example_df = create_lag_features_for_training_only(train_df)
display(lag_feature_example_df.head(24 * 7 + 3))

## Machine Learning

In [ ]:
model_dict = {}


def lightgbm_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.lgb_params, "seed": seed}
    lgb_train = lgb.Dataset(x_train, y_train, categorical_feature=categorical_features)
    lgb_valid = lgb.Dataset(x_valid, y_valid, categorical_feature=categorical_features)
    model = lgb.train(
        params=params,
        train_set=lgb_train,
        num_boost_round=CFG.num_boost_round,
        valid_sets=[lgb_train, lgb_valid],
        callbacks=[
            lgb.early_stopping(stopping_rounds=CFG.early_stopping_round, verbose=CFG.verbose),
            lgb.log_evaluation(CFG.verbose),
        ],
    )
    # Predict validation
    valid_pred = model.predict(x_valid)
    return model, valid_pred


def xgboost_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.xgb_params, "seed": seed}
    xgb_train = xgb.DMatrix(data=x_train, label=y_train, enable_categorical=True)
    xgb_valid = xgb.DMatrix(data=x_valid, label=y_valid, enable_categorical=True)
    model = xgb.train(
        params,
        dtrain=xgb_train,
        num_boost_round=CFG.num_boost_round,
        evals=[(xgb_train, "train"), (xgb_valid, "eval")],
        early_stopping_rounds=CFG.early_stopping_round,
        verbose_eval=CFG.verbose,
    )
    # Predict validation
    valid_pred = model.predict(xgb.DMatrix(x_valid, enable_categorical=True))
    return model, valid_pred


def catboost_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    params = {**CFG.cat_params, "random_seed": seed}
    cat_train = Pool(data=x_train, label=y_train, cat_features=categorical_features)
    cat_valid = Pool(data=x_valid, label=y_valid, cat_features=categorical_features)
    model = CatBoostRegressor(**params)
    model.fit(
        cat_train,
        eval_set=[cat_valid],
        early_stopping_rounds=CFG.early_stopping_round,
        verbose=CFG.verbose,
        use_best_model=True,
    )
    # Predict validation
    valid_pred = model.predict(x_valid)
    return model, valid_pred


def linear_training(
    x_train: pd.DataFrame,
    y_train: pd.DataFrame,
    x_valid: pd.DataFrame,
    y_valid: pd.DataFrame,
    features: list,
    categorical_features: list,
    seed: int,
):
    numeric_features = [col for col in features if col not in categorical_features]
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", Ridge(alpha=10.0)),
        ]
    )
    model.fit(x_train, y_train)
    valid_pred = model.predict(x_valid)

    return model, valid_pred


def gradient_boosting_model_cv_training(
    method: str,
    train_df: pd.DataFrame,
    features: list,
    categorical_features: list,
    linear_categorical_features: list,
    seed: int,
):
    # Create a numpy array to store out of folds predictions
    oof_predictions = np.full(len(train_df), np.nan)
    oof_fold = np.zeros(len(train_df))
    target_col = CFG.target_col
    seed_everything(seed)

    # 時系列では validation より未来のデータを学習に使わない
    splits = make_time_series_splits(train_df)

    for fold, (train_index, valid_index) in enumerate(splits):
        print("-" * 50)
        print(f"{method} (seed {seed}) training fold {fold+1}")
        x_train = train_df[features].iloc[train_index]
        y_train = train_df[target_col].iloc[train_index]
        x_valid = train_df[features].iloc[valid_index]
        y_valid = train_df[target_col].iloc[valid_index]
        if method == "lightgbm":
            model, valid_pred = lightgbm_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
            importance_df = pd.DataFrame(
                model.feature_importance(), index=features, columns=["importance"]
            ).reset_index()
            importance_df.to_csv(
                CFG.MODEL_DATA_PATH
                / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}_importance.csv",
                index=False,
            )
        if method == "xgboost":
            model, valid_pred = xgboost_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
        if method == "catboost":
            model, valid_pred = catboost_training(
                x_train, y_train, x_valid, y_valid, features, categorical_features, seed
            )
        if method == "linear":
            model, valid_pred = linear_training(
                x_train,
                y_train,
                x_valid,
                y_valid,
                features,
                linear_categorical_features,
                seed,
            )
        # Save best model
        with open(
            CFG.MODEL_DATA_PATH
            / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}.pkl",
            "wb",
        ) as f:
            pickle.dump(model, f)
        model_dict[f"{method}_{target_col}_fold{fold+1}_seed{seed}"] = model

        # Add to out of folds array
        oof_predictions[valid_index] = valid_pred
        del x_train, x_valid, y_train, y_valid, model, valid_pred
        gc.collect()
        oof_fold[valid_index] = fold + 1

    oof_df = pd.DataFrame(
        {
            CFG.target_col: oof_predictions.astype(float),
            "fold": oof_fold.astype(int),
            "row_id": train_df.index,
            CFG.datetime_col: train_df[CFG.datetime_col].astype(str),
        }
    )

    valid_mask = np.isfinite(oof_predictions)
    y_true = train_df.loc[valid_mask, CFG.target_col].values
    score = calc_score(y_true, oof_predictions[valid_mask])
    print(f"{method} (seed {seed}) our out of folds CV {CFG.metric} is {score}")
    print(f"OOF coverage: {valid_mask.mean():.1%}")

    oof_path = CFG.OOF_DATA_PATH / f"oof_{CFG.AUTHOR}_{method}_seed{seed}_ver{CFG.VER}.csv"
    oof_df.to_csv(oof_path, index=False)

    return score

In [ ]:
for method in CFG.METHOD_LIST:
    scores = []
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        print("=" * 50)
        print(f"Training {method} with seed {seed} ({seed_idx + 1}/{CFG.N_SEEDS})")
        print("=" * 50)
        score = gradient_boosting_model_cv_training(
            method,
            train_full_df,
            features,
            categorical_features,
            linear_categorical_features,
            seed,
        )
        scores.append(score)

    print("=" * 50)
    print(f"{method} Seed Averaging Results:")
    print(f"  Seeds: {[CFG.SEED + i for i in range(CFG.N_SEEDS)]}")
    print(f"  Scores: {scores}")
    print(f"  Mean: {np.mean(scores):.6f}")
    print(f"  Std: {np.std(scores):.6f}")
    print("=" * 50)

In [ ]:
def load_model(method: str, target_col: str, fold: int, seed: int):
    with open(
        CFG.MODEL_DATA_PATH
        / f"{CFG.AUTHOR}_{method}_{target_col}_fold{fold + 1}_seed{seed}_ver{CFG.VER}.pkl",
        "rb",
    ) as f:
        return pickle.load(f)


def lightgbm_inference(x_test: pd.DataFrame, target_col: str):
    test_pred = np.zeros(len(x_test))
    total_models = 0
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        for fold in range(CFG.N_SPLIT):
            model = load_model("lightgbm", target_col, fold, seed)
            # Predict
            test_pred += model.predict(x_test)
            total_models += 1
    return test_pred / total_models


def xgboost_inference(x_test: pd.DataFrame, target_col: str):
    test_pred = np.zeros(len(x_test))
    total_models = 0
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        for fold in range(CFG.N_SPLIT):
            model = load_model("xgboost", target_col, fold, seed)
            # Predict
            test_pred += model.predict(xgb.DMatrix(x_test, enable_categorical=True))
            total_models += 1
    return test_pred / total_models


def catboost_inference(x_test: pd.DataFrame, target_col: str):
    test_pred = np.zeros(len(x_test))
    total_models = 0
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        for fold in range(CFG.N_SPLIT):
            model = load_model("catboost", target_col, fold, seed)
            # Predict
            test_pred += model.predict(x_test)
            total_models += 1
    return test_pred / total_models


def linear_inference(x_test: pd.DataFrame, target_col: str):
    test_pred = np.zeros(len(x_test))
    total_models = 0
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        for fold in range(CFG.N_SPLIT):
            model = load_model("linear", target_col, fold, seed)
            test_pred += model.predict(x_test)
            total_models += 1
    return test_pred / total_models


def gradient_boosting_model_inference(
    method: str, target_col: str, test_df: pd.DataFrame, features: list
):
    x_test = test_df[features]
    if method == "lightgbm":
        test_pred = lightgbm_inference(x_test, target_col)
    if method == "xgboost":
        test_pred = xgboost_inference(x_test, target_col)
    if method == "catboost":
        test_pred = catboost_inference(x_test, target_col)
    if method == "linear":
        test_pred = linear_inference(x_test, target_col)
    return test_pred


def predicting(input_df: pd.DataFrame, features: list):
    output_df = input_df.copy()
    target_col = CFG.target_col
    print(f"{target_col} inference")
    print(
        f"Using {CFG.N_SEEDS} seeds x {CFG.N_SPLIT} folds "
        f"= {CFG.N_SEEDS * CFG.N_SPLIT} models per method"
    )
    output_df[target_col] = 0.0
    weight_sum = sum(CFG.METHOD_WEIGHT_DICT.values())
    for method in CFG.METHOD_LIST:
        output_df[f"{method}_{target_col}_pred"] = gradient_boosting_model_inference(
            method, target_col, input_df, features
        )
        output_df[target_col] += (
            output_df[f"{method}_{target_col}_pred"]
            * CFG.METHOD_WEIGHT_DICT[method]
            / weight_sum
        )
    return output_df

In [ ]:
def average_predictions_with_nan(predictions):
    predictions = np.asarray(predictions, dtype=float)
    valid_counts = np.sum(np.isfinite(predictions), axis=0)
    return np.divide(
        np.nansum(predictions, axis=0),
        valid_counts,
        out=np.full(predictions.shape[1], np.nan),
        where=valid_counts > 0,
    )


def load_oof_preds(method: str, train_df: pd.DataFrame) -> np.ndarray:
    # OOFファイル群を読み込み、train_df.index に沿った平均予測を返す
    per_seed_preds = []
    for seed_idx in range(CFG.N_SEEDS):
        seed = CFG.SEED + seed_idx
        path = CFG.OOF_DATA_PATH / f"oof_{CFG.AUTHOR}_{method}_seed{seed}_ver{CFG.VER}.csv"
        oof = pd.read_csv(path)
        if "row_id" not in oof.columns:
            raise ValueError(
                f"OOF file for {method} has no 'row_id'. 学習側で row_id を保存してください。"
            )
        ordered = oof.set_index("row_id").loc[train_df.index]
        per_seed_preds.append(ordered[CFG.target_col].astype(float).values)

    if not per_seed_preds:
        raise ValueError(f"No OOF files found for {method}")

    return average_predictions_with_nan(per_seed_preds)


def eval_single_oof_from_file(method: str, train_df: pd.DataFrame) -> dict:
    # 単一メソッドの OOF を読み込んで指標を計算
    y_pred = load_oof_preds(method, train_df)
    valid_mask = np.isfinite(y_pred)
    metrics = {
        "method": method,
        "coverage": valid_mask.mean(),
        **calculate_regression_metrics(
            train_df.loc[valid_mask, CFG.target_col].values,
            y_pred[valid_mask],
        ),
    }
    print(f"[{method}] {metrics}")
    return metrics


def eval_blend_oof_from_files(train_df: pd.DataFrame, method_list: list) -> tuple:
    # 複数メソッドの OOF を重み付きブレンドして指標を計算
    weights = np.array([CFG.METHOD_WEIGHT_DICT[m] for m in method_list], dtype=float)
    weights = weights / weights.sum()
    pred_matrix = np.vstack([load_oof_preds(m, train_df) for m in method_list])
    blended_pred = np.nansum(pred_matrix * weights[:, np.newaxis], axis=0)
    blended_pred[np.any(~np.isfinite(pred_matrix), axis=0)] = np.nan

    valid_mask = np.isfinite(blended_pred)
    metrics = {
        "method": f"blend({','.join(method_list)})",
        "coverage": valid_mask.mean(),
        **calculate_regression_metrics(
            train_df.loc[valid_mask, CFG.target_col].values,
            blended_pred[valid_mask],
        ),
    }
    print(f"[BLEND] {metrics}")

    return metrics, blended_pred

In [ ]:
metric_rows = []
for m in CFG.METHOD_LIST:
    metric_rows.append(eval_single_oof_from_file(m, train_full_df))

blend_metrics, oof_pred = eval_blend_oof_from_files(train_full_df, CFG.METHOD_LIST)
metric_rows.append(blend_metrics)

metric_df = pd.DataFrame(metric_rows).sort_values(CFG.metric)
display(metric_df)

In [ ]:
test_pred_df = predicting(feat_test_df, features)

In [ ]:
submission_df[CFG.raw_target_col] = to_original_scale(test_pred_df[CFG.target_col].values)
submission_df.to_csv(CFG.OOF_DATA_PATH / "submission.csv", index=False)
submission_df

## Feature importance

In [ ]:
# 全シードの平均importanceを計算
all_importances = []
for seed_idx in range(CFG.N_SEEDS):
    seed = CFG.SEED + seed_idx
    for fold in range(CFG.N_SPLIT):
        model = load_model("lightgbm", CFG.target_col, fold, seed)
        all_importances.append(model.feature_importance())

mean_importance = np.mean(all_importances, axis=0)
std_importance = np.std(all_importances, axis=0)

importance_df = pd.DataFrame({
    "feature": feat_train_df.columns,
    "importance": mean_importance,
    "std": std_importance,
}).sort_values("importance", ascending=False).head(50)

fig, ax = plt.subplots(figsize=(8, max(6, len(importance_df) * 0.25)))
ax.barh(importance_df["feature"], importance_df["importance"], xerr=importance_df["std"])
ax.invert_yaxis()
ax.set_xlabel("Feature Importance")
ax.set_title(f"Feature Importance (Mean over {CFG.N_SEEDS} seeds x {CFG.N_SPLIT} folds)")
ax.grid()
fig.tight_layout()
plt.show()

## Time Series Prediction Evaluation

In [ ]:
def create_oof_diagnostics(input_df, y_pred):
    valid_mask = np.isfinite(y_pred)
    diagnostics_df = input_df.loc[
        valid_mask,
        [CFG.datetime_col, CFG.raw_target_col],
    ].copy()
    diagnostics_df["prediction"] = to_original_scale(y_pred[valid_mask])
    diagnostics_df["residual"] = (
        diagnostics_df["prediction"] - diagnostics_df[CFG.raw_target_col]
    )
    diagnostics_df["abs_error"] = diagnostics_df["residual"].abs()
    diagnostics_df["squared_error"] = diagnostics_df["residual"] ** 2
    diagnostics_df["hour"] = diagnostics_df[CFG.datetime_col].dt.hour
    diagnostics_df["dayofweek"] = diagnostics_df[CFG.datetime_col].dt.dayofweek
    diagnostics_df["month"] = diagnostics_df[CFG.datetime_col].dt.to_period("M").astype(str)
    return diagnostics_df


def summarize_error_by_group(diagnostics_df, group_col):
    return (
        diagnostics_df.groupby(group_col)
        .agg(
            samples=(CFG.raw_target_col, "size"),
            actual_mean=(CFG.raw_target_col, "mean"),
            prediction_mean=("prediction", "mean"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda values: np.sqrt(values.mean())),
            mean_error=("residual", "mean"),
        )
        .reset_index()
    )


diagnostics_df = create_oof_diagnostics(train_df, oof_pred)
display(diagnostics_df.sort_values("abs_error", ascending=False).head(20))

In [ ]:
def plot_daily_oof_predictions(diagnostics_df):
    daily_df = (
        diagnostics_df.set_index(CFG.datetime_col)[[CFG.raw_target_col, "prediction"]]
        .resample("D")
        .sum()
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(daily_df[CFG.datetime_col], daily_df[CFG.raw_target_col], label="actual")
    ax.plot(daily_df[CFG.datetime_col], daily_df["prediction"], label="OOF prediction")
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily rental count")
    ax.set_title("Daily Actual vs OOF Prediction")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)
    fig.tight_layout()
    plt.show()


plot_daily_oof_predictions(diagnostics_df)

In [ ]:
def plot_error_by_group(diagnostics_df):
    hour_df = summarize_error_by_group(diagnostics_df, "hour")
    dayofweek_df = summarize_error_by_group(diagnostics_df, "dayofweek")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    sns.barplot(data=hour_df, x="hour", y="mae", ax=axes[0])
    axes[0].set_title("MAE by Hour")
    axes[0].grid(True, axis="y", linestyle="--", alpha=0.5)

    sns.barplot(data=dayofweek_df, x="dayofweek", y="mae", ax=axes[1])
    axes[1].set_title("MAE by Day of Week (0=Monday)")
    axes[1].grid(True, axis="y", linestyle="--", alpha=0.5)
    fig.tight_layout()
    plt.show()

    return hour_df, dayofweek_df


hour_error_df, dayofweek_error_df = plot_error_by_group(diagnostics_df)
display(hour_error_df)
display(dayofweek_error_df)

In [ ]:
def plot_residual_diagnostics(diagnostics_df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    sns.histplot(diagnostics_df["residual"], bins=50, ax=axes[0])
    axes[0].set_title("OOF Residual Distribution")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    axes[1].scatter(
        diagnostics_df[CFG.raw_target_col],
        diagnostics_df["prediction"],
        s=8,
        alpha=0.3,
    )
    max_value = max(
        diagnostics_df[CFG.raw_target_col].max(),
        diagnostics_df["prediction"].max(),
    )
    axes[1].plot([0, max_value], [0, max_value], color="red", linestyle="--")
    axes[1].set_xlabel("Actual count")
    axes[1].set_ylabel("OOF prediction")
    axes[1].set_title("Actual vs OOF Prediction")
    axes[1].grid(True, linestyle="--", alpha=0.5)
    fig.tight_layout()
    plt.show()


plot_residual_diagnostics(diagnostics_df)